# Harmonization fix + re-statistics (run for BOTH pipelines)

**Why:** in the previous run, ComBat produced an all-NaN matrix (a site with 1 subject,
`CMU`, breaks neuroCombat) and the notebook silently fell back to **raw, un-harmonised**
features. This notebook fixes that and recomputes the statistics on genuinely harmonised data.

**You do NOT need to re-extract features.** This loads your saved `features_raw_*.csv`,
so it runs in ~1 minute.

**Run it twice**, setting the parameters in the first cell each time:
1. `RAW_FEATURES_CSV` = your **no-GSR** raw features CSV, `PIPELINE='noGSR'`.
2. `RAW_FEATURES_CSV` = your **GSR** raw features CSV, `PIPELINE='GSR'`.

The key change vs your original code: sites with fewer than `MIN_SITE_N` subjects are
dropped before ComBat, and **ComBat now fails loudly** (raises) instead of silently using
raw data — so this class of bug can never hide again.


In [ ]:
!pip install neuroCombat nibabel nilearn networkx seaborn xgboost -q

# ============================ PARAMETERS ============================
RAW_FEATURES_CSV = '/kaggle/input/datasets/saeedrezaeiafshar/gsr-nogsr-abide-cbt-results/GSR_results/features_raw_GSR.csv'   # <- set per run
PIPELINE         = 'GSR'          # 'noGSR' or 'GSR' (used only for output names)
MIN_SITE_N       = 3                # drop sites with fewer than this many subjects
                                    #   CMU has 1 and UCLA_1 has 2 in the no-GSR sample.
                                    #   Start at 3; if you want to keep UCLA_1, try 2 and
                                    #   confirm the ComBat assertion below still passes.
PHENO_URL = 'https://s3.amazonaws.com/fcp-indi/data/Projects/ABIDE_Initiative/Phenotypic_V1_0b_preprocessed1.csv'
OUTDIR = '/kaggle/working/results'
# ===================================================================

import os, numpy as np, pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
from neuroCombat import neuroCombat
os.makedirs(OUTDIR, exist_ok=True)

def adaptive_test_full(g1, g2, alpha=0.05):
    g1 = np.asarray(g1, float); g2 = np.asarray(g2, float)
    g1 = g1[~np.isnan(g1)]; g2 = g2[~np.isnan(g2)]
    if len(g1) < 3 or len(g2) < 3:
        return ('skip', np.nan, np.nan)
    sw1, sp1 = stats.shapiro(g1); sw2, sp2 = stats.shapiro(g2)
    lev_stat, lev_p = stats.levene(g1, g2)
    if sp1 > alpha and sp2 > alpha:
        stat, p = stats.ttest_ind(g1, g2, equal_var=(lev_p > alpha))
        name = 't-test' if lev_p > alpha else 't-test (Welch)'
    else:
        stat, p = stats.mannwhitneyu(g1, g2, alternative='two-sided')
        name = 'Mann-Whitney U'
    return (name, stat, p)

raw = pd.read_csv(RAW_FEATURES_CSV)
meta_cols = [c for c in ['Subject','Group','Center','IQ','FIQ'] if c in raw.columns]
feat_cols = [c for c in raw.columns if c not in meta_cols]
print(f"Loaded {RAW_FEATURES_CSV}: {raw.shape}  | {len(feat_cols)} features")
print("Site sizes:\n", raw['Center'].value_counts().sort_values().to_string())



In [ ]:

# ---- drop under-populated sites, then run ComBat (loud) ----
site_counts = raw['Center'].value_counts()
drop_sites = site_counts[site_counts < MIN_SITE_N].index.tolist()
kept = raw[~raw['Center'].isin(drop_sites)].reset_index(drop=True).copy()
print(f"Dropping sites with <{MIN_SITE_N} subjects: {drop_sites}")
print(f"After site filter: {len(kept)} "
      f"(ASD={sum(kept['Group']=='ASD')}, TD={sum(kept['Group']=='Control')})")

site_encoder = {s:i for i,s in enumerate(kept['Center'].unique())}
covars = pd.DataFrame({
    'SITE':  kept['Center'].map(site_encoder).values,
    'Group': kept['Group'].map({'ASD':1,'Control':0}).values,
    'IQ':    kept['IQ'].fillna(kept['IQ'].median()).values if 'IQ' in kept else 0.0,
})
X = kept[feat_cols].values.astype(np.float64)
for j in range(X.shape[1]):                      # median-fill any feature NaNs
    col = X[:, j]; col[np.isnan(col)] = np.nanmedian(col); X[:, j] = col

res = neuroCombat(dat=X.T, covars=covars, batch_col='SITE')['data']  # features x subjects
nan_frac = np.isnan(res).mean()
assert nan_frac < 1e-6, (
    f"ComBat still produced NaNs (fraction={nan_frac:.3f}). "
    f"Increase MIN_SITE_N (currently {MIN_SITE_N}) and re-run.")
print(f"ComBat OK — output NaN fraction = {nan_frac:.6f}")

harmonized_df = kept[meta_cols].copy()
for i, c in enumerate(feat_cols):
    harmonized_df[c] = res[i, :]
if 'IQ' in harmonized_df and 'FIQ' not in harmonized_df:
    harmonized_df['FIQ'] = harmonized_df['IQ']
harmonized_df.to_csv(f'{OUTDIR}/features_ComBat_{PIPELINE}_FIXED.csv', index=False)

# sanity: harmonised must now DIFFER from raw
diff = np.nanmax(np.abs(harmonized_df[feat_cols].values -
                        kept[feat_cols].values))
print(f"Max |harmonised - raw| = {diff:.4f}  (must be > 0; was 0 in the broken run)")



In [ ]:

# ---- whole / intra / inter group stats with per-family FDR ----
asd  = harmonized_df[harmonized_df['Group']=='ASD']
ctrl = harmonized_df[harmonized_df['Group']=='Control']
cats = {'Whole':[], 'Intra':[], 'Inter':[]}
for c in feat_cols:
    if c.startswith('Whole_'): cats['Whole'].append(c)
    elif c.startswith('Intra_'): cats['Intra'].append(c)
    elif c.startswith('Inter_'): cats['Inter'].append(c)

rows=[]
for cat, fl in cats.items():
    sub=[]
    for f in fl:
        name, stat, p = adaptive_test_full(asd[f].values, ctrl[f].values)
        if name=='skip': continue
        sub.append({'Category':cat,'Feature':f,'Test':name,
                    'ASD_Mean':asd[f].mean(),'TD_Mean':ctrl[f].mean(),
                    'ASD_SD':asd[f].std(),'TD_SD':ctrl[f].std(),'p':p})
    d=pd.DataFrame(sub)
    if len(d):
        d['p_FDR']=multipletests(d['p'], method='fdr_bh')[1]
        rows.append(d)
stat_df=pd.concat(rows, ignore_index=True).sort_values(['Category','p'])
stat_df.to_csv(f'{OUTDIR}/statistical_comparison_{PIPELINE}_FIXED.csv', index=False)

print(f"\n================  {PIPELINE}  (HARMONISED)  ================")
print("WHOLE-BRAIN family:")
print(stat_df[stat_df.Category=='Whole'][
    ['Feature','ASD_Mean','TD_Mean','Test','p','p_FDR']].to_string(index=False))
for cat in ['Intra','Inter']:
    d=stat_df[stat_df.Category==cat]
    sig=d[d.p<0.05]
    print(f"\n{cat}: {len(d)} tested | uncorrected p<0.05: {len(sig)} | FDR<0.05: {(d.p_FDR<0.05).sum()}")
    if len(sig): print(sig[['Feature','ASD_Mean','TD_Mean','p','p_FDR']].head(8).to_string(index=False))



In [ ]:

# ---- behavioural correlations + PLSR + PLSC + H-vs-metrics on HARMONISED data ----
from numpy.linalg import svd
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import cross_val_score

pheno = pd.read_csv(PHENO_URL)
BEHAV = {'ADI_R_SOCIAL_TOTAL_A':'ADI-R Social','ADI_R_VERBAL_TOTAL_BV':'ADI-R Verbal',
         'ADI_R_ONSET_TOTAL_D':'ADI-R Onset','ADOS_TOTAL':'ADOS Total','ADOS_COMM':'ADOS Communication'}
bdf = pheno[['SUB_ID']+list(BEHAV)].rename(columns=BEHAV); bdf['Subject']=bdf['SUB_ID']
for c in BEHAV.values():
    bdf[c]=pd.to_numeric(bdf[c],errors='coerce'); bdf.loc[bdf[c]<0,c]=np.nan
behav_cols=list(BEHAV.values())
am = asd.merge(bdf[['Subject']+behav_cols], on='Subject', how='left')
net = [c for c in feat_cols]

# (a) pairwise correlations
cr=[]
for b in behav_cols:
    s=am[['Subject',b]+net].dropna(subset=[b])
    if len(s)<10: continue
    for f in net:
        pr=s[[f,b]].dropna()
        if len(pr)<10: continue
        rr,pp=stats.pearsonr(pr[f],pr[b]); rho,sp=stats.spearmanr(pr[f],pr[b])
        cr.append({'Feature':f,'Score':b,'n':len(pr),'r':rr,'p':pp,'rho':rho,'sp':sp})
cdf=pd.DataFrame(cr)
if len(cdf):
    cdf['p_FDR']=multipletests(cdf['p'],method='fdr_bh')[1]
    cdf=cdf.sort_values('p'); cdf.to_csv(f'{OUTDIR}/behavioral_correlations_{PIPELINE}_FIXED.csv',index=False)
    print("Behavioural pairs:",len(cdf),"| nominal p<0.05:",int((cdf.p<0.05).sum()),
          "| FDR<0.05:",int((cdf.p_FDR<0.05).sum()))
    print("\nWhole_Energy_global vs behaviour:")
    print(cdf[cdf.Feature=='Whole_Energy_global'][['Score','n','r','p','rho','sp']].to_string(index=False))

# (b) PLSR cross-validated R2
print("\nPLSR cross-validated R^2:")
for b in behav_cols:
    s=am[['Subject',b]+net].dropna(subset=[b])
    fe=s[net].dropna(axis=1,thresh=int(0.8*len(s))).fillna(s[net].median())
    if len(s)<20 or fe.shape[1]<2: print(f"  {b}: skip (n={len(s)})"); continue
    Xp=StandardScaler().fit_transform(fe.values); yp=(s[b].values-s[b].mean())/s[b].std()
    nc=min(3,Xp.shape[1],Xp.shape[0]-1)
    cv=cross_val_score(PLSRegression(n_components=1,scale=False),Xp,yp,cv=5,scoring='r2').mean()
    print(f"  {b}: n={len(s)}, 1-comp CV R2={cv:.3f}")

# (c) PLSC with permutation test
def plsc(Xdf,Ydf,n_perm=5000,seed=0):
    rng=np.random.default_rng(seed); Xx=Xdf.values.astype(float); Yy=Ydf.values.astype(float)
    Xz=(Xx-Xx.mean(0))/(Xx.std(0,ddof=1)+1e-12); Yz=(Yy-Yy.mean(0))/(Yy.std(0,ddof=1)+1e-12)
    n=len(Xz); R=Yz.T@Xz/(n-1); U,S,Vt=svd(R,full_matrices=False)
    ge=np.zeros(len(S))
    for _ in range(n_perm):
        sp=svd(Yz[rng.permutation(n)].T@Xz/(n-1),compute_uv=False); m=min(len(sp),len(S)); ge[:m]+=sp[:m]>=S[:m]
    return S, S**2/np.sum(S**2), (ge+1)/(n_perm+1), n
for label, cols in [('ADOS_block',['ADOS Total','ADOS Communication']),('Full_block',behav_cols)]:
    cc=[c for c in cols if c in am.columns]; blk=am[['Subject']+net+cc].dropna(subset=cc)
    fe=[c for c in net if blk[c].notna().mean()>0.8]; blk[fe]=blk[fe].fillna(blk[fe].median()); blk=blk.dropna(subset=fe)
    if len(blk)<20 or len(fe)<2: print(f"PLSC {label}: skip (n={len(blk)})"); continue
    S,ce,pv,n=plsc(blk[fe],blk[cc])
    print(f"PLSC {label} (n={n}): LV1 cov={ce[0]*100:.1f}% perm p={pv[0]:.4f}; LV2 perm p={pv[1] if len(pv)>1 else float('nan'):.4f}")

# (d) Hamiltonian vs other metrics (within group), FDR per group
print("\nHamiltonian vs metrics (top correlates):")
for gname,gdf in [('ASD',asd),('TD',ctrl)]:
    hr=[]
    for f in net:
        if f=='Whole_Energy_global': continue
        d=gdf[['Whole_Energy_global',f]].dropna()
        if len(d)<10: continue
        rr,pp=stats.pearsonr(d['Whole_Energy_global'],d[f]); hr.append({'Feature':f,'r':rr,'p':pp})
    h=pd.DataFrame(hr)
    if len(h):
        h['p_FDR']=multipletests(h['p'],method='fdr_bh')[1]; h=h.sort_values('p')
        h.to_csv(f'{OUTDIR}/hamiltonian_vs_metrics_{PIPELINE}_FIXED_{gname}.csv',index=False)
        eff=h[h.Feature.str.contains('Efficiency')]
        print(f"  [{gname}] efficiency vs H:")
        print(eff[['Feature','r','p','p_FDR']].to_string(index=False))



In [ ]:

# ---- signed modularity re-stats on HARMONISED H / efficiency (reuse saved Q values) ----
import os
MODCSV=f'{OUTDIR}/signed_modularity_{PIPELINE}.csv'
if not os.path.exists(MODCSV): MODCSV=f'{OUTDIR}/signed_modularity_noGSR.csv'
if os.path.exists(MODCSV):
    mod=pd.read_csv(MODCSV)[['Subject','SignedModularity']].dropna()
    m=harmonized_df[['Subject','Group','Whole_Energy_global',
                     'Whole_GlobalEfficiency','Whole_LocalEfficiency']].merge(mod,on='Subject',how='inner')
    a=m[m.Group=='ASD']['SignedModularity']; t=m[m.Group=='Control']['SignedModularity']
    nm,st,p=adaptive_test_full(a.values,t.values)
    print(f"Signed modularity (harmonised cohort): ASD={a.mean():.4f}, TD={t.mean():.4f}, {nm} p={p:.4f}")
    for g,l in [('ASD','ASD'),('Control','TD')]:
        d=m[m.Group==g].dropna(subset=['SignedModularity','Whole_Energy_global'])
        rH,pH=stats.pearsonr(d.SignedModularity,d.Whole_Energy_global)
        rE,pE=stats.pearsonr(d.SignedModularity,d.Whole_GlobalEfficiency)
        print(f"  [{l}] Q vs H r={rH:.3f} p={pH:.4f} | Q vs global-eff r={rE:.3f} p={pE:.4f} (n={len(d)})")
else:
    print("signed_modularity CSV not found; re-run the modularity add-on cell if you want this updated.")
print("\nDONE. Send me the printed WHOLE-BRAIN table + behavioural/PLSC/modularity lines for this pipeline.")



## What to send back

For **each** pipeline (GSR and no-GSR): the harmonised WHOLE-BRAIN table, the intra/inter
significant-count lines, the `Whole_Energy_global` behaviour correlations, the PLSR R^2 and
PLSC LV1 perm-p lines, and the signed-modularity lines. With genuinely harmonised numbers in
hand I will update the manuscript tables/text so everything is accurate and consistent.

Also confirm whether your **original GSR run** showed the same `using raw data` warning —
that tells us whether the primary results need updating too.
